# Práctica: cifrado César sobre una red simulada

Este notebook explica, de principio a fin, cómo funciona nuestra práctica: el algoritmo de cifrado, la topología de red donde vive, por qué ambos extremos necesitan la misma llave, el direccionamiento IP, las conexiones físicas, y cómo se sirven y se buscan las páginas desde cada PC.

**Índice**
1. ¿Qué es el cifrado César?
2. La fórmula, paso a paso
3. Cifrar y descifrar con el mismo mecanismo
4. Por qué ambos extremos deben conocer la llave
5. Topología de red de la práctica
6. Direccionamiento IP
7. Conexiones físicas y tipos de cable
8. Qué hace cada PC
9. Cómo se sirven las páginas y cómo se buscan por IP
10. Control de acceso (ACL)
11. Resumen de la práctica


## 1. ¿Qué es el cifrado César?

Es uno de los cifrados más antiguos que existen: cada letra del mensaje se reemplaza por otra letra que está un número fijo de posiciones más adelante en el alfabeto. Ese número fijo es la **llave** (`k`).

Por ejemplo, con `k = 3`, la letra `a` se convierte en `d`, la `b` en `e`, y así sucesivamente. Cuando se llega al final del alfabeto, se vuelve a empezar desde la `a` — por eso el algoritmo necesita una operación de módulo (`% 26`).

Los espacios, números y signos de puntuación no se tocan; solo se desplazan las letras.

In [ ]:
def cc(t_p, k):
    # Cadena vacía para almacenar el resultado cifrado
    t_c = ""

    # Recorremos cada carácter del texto plano (t_p)
    for caracter in t_p:
        # Verificamos si el carácter es una letra alfabética
        if caracter.isalpha():
            # Obtenemos el valor numérico ASCII de la letra base 'a' (97)
            codigo_base = ord('a')

            # Aplicamos la fórmula matemática del cifrado César:
            # 1. ord(caracter) - codigo_base: convierte la letra a rango 0-25
            # 2. + k: suma el desplazamiento
            # 3. % 26: asegura el ciclo en el alfabeto
            # 4. + codigo_base: devuelve el número al rango ASCII de minúsculas
            # 5. chr(...): convierte el número final de vuelta a carácter
            formula = chr((ord(caracter) - codigo_base + k) % 26 + codigo_base)

            # Agregamos la letra cifrada al resultado
            t_c += formula
        else:
            # Si es espacio, número o puntuación, se mantiene igual
            t_c += caracter

    return t_c


# Ejemplo rápido
mensaje = "hola mundo"
llave = 3
cifrado = cc(mensaje, llave)

print("Texto original:", mensaje)
print("Llave:", llave)
print("Texto cifrado:", cifrado)


## 2. La fórmula, paso a paso

Para una letra cualquiera y una llave `k`, el cifrado hace esto:

```
posición = ord(letra) - ord('a')       # la 'a' pasa a ser 0, la 'b' pasa a ser 1, ...
nueva_posición = (posición + k) % 26   # se suma el desplazamiento y se ajusta el ciclo
nueva_letra = chr(nueva_posición + ord('a'))
```

El `% 26` es la parte que hace que el alfabeto se comporte como un círculo: si te pasas de la `z`, regresas al principio. Por ejemplo, con `k = 3`, la letra `x` no se convierte en un carácter fuera del alfabeto, sino en `a`:

```
posición de 'x' = 23
23 + 3 = 26
26 % 26 = 0   ->  'a'
```

Nota: la fórmula original del profesor siempre usa `ord('a')` como base, así que incluso una letra mayúscula termina cifrada como una letra minúscula. Es una particularidad del código, no un error nuestro — la reproducimos igual en la página web para que el resultado coincida.

In [ ]:
# Comprobamos ese caso límite
print(cc("x", 3))   # debe imprimir 'a'
print(cc("Hola", 3)) # la H mayúscula también se cifra con base 'a'


## 3. Cifrar y descifrar con el mismo mecanismo

El cifrado César es simétrico: para descifrar se usa exactamente la misma función, pero con la llave en negativo. Si cifrar es "avanzar k posiciones", descifrar es "retroceder k posiciones" — y retroceder es lo mismo que avanzar `-k`.

In [ ]:
def descifrar(texto_cifrado, k):
    return cc(texto_cifrado, -k)


mensaje = "hola mundo"
llave = 3

cifrado = cc(mensaje, llave)
descifrado = descifrar(cifrado, llave)

print("Original:  ", mensaje)
print("Cifrado:   ", cifrado)
print("Descifrado:", descifrado)
print("¿Coincide con el original?", descifrado == mensaje)


## 4. Por qué ambos extremos deben conocer la llave

El cifrado César es de **llave simétrica**: la misma llave que se usó para cifrar es la única que puede descifrar correctamente. Si PC1 cifra con `k = 3` y PC2 intenta descifrar con cualquier otra llave, el resultado es basura, no el mensaje original.

Por eso la llave **no viaja dentro del mensaje cifrado** ni se resuelve "adivinando": los dos lados tienen que acordarla de antemano, por un canal aparte (de palabra, en un documento, como parte de la práctica). Esto es justo lo que hace que nuestra página de PC2 pida la llave además del texto cifrado, en vez de mostrar el mensaje solo con el texto.

In [ ]:
# Qué pasa si PC2 usa una llave equivocada
mensaje = "reunion a las 5"
llave_correcta = 7

cifrado = cc(mensaje, llave_correcta)
print("Cifrado enviado:", cifrado)

for llave_intentada in [3, 5, 7, 10]:
    resultado = cc(cifrado, -llave_intentada)
    marca = "correcta" if llave_intentada == llave_correcta else "incorrecta"
    print(f"Llave {llave_intentada} ({marca}) -> {resultado}")


## 5. Topología de red de la práctica

```
PC1 --- Switch1 --- Router1 === [ NUBE / enlace WAN ] === Router2 --- Switch2 --- PC2
(cifra)   2960        1941                                    1941        2960    (visualiza)
```

- **PC1** es donde se cifra el mensaje. Corre la página completa (`pc1_cifrado.html`), con el campo de la llave y la opción de cifrar/descifrar. Esta página **no se sirve por red** — se abre localmente en PC1, para que no haya forma de que alguien la alcance por accidente desde la otra LAN.
- **PC2** es donde se visualiza el mensaje. Corre la página de solo lectura (`pc2_mensaje.html`), que pide el texto cifrado recibido y la llave ya acordada, y hace el descifrado en el propio navegador.
- **Router1 — Router2** representan el enlace que en el diagrama de la práctica aparece como "nube": en el laboratorio físico no hay ninguna nube real, ese tramo se simula con un cable serial (V.35, con un lado DCE y otro DTE) si los routers tienen módulo WIC-2T, o con un cable Ethernet directo si no lo tienen.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(9, 3.2))

cajas = [
    ("PC1\n(cifra)", 0.5, "#f0997b"),
    ("Switch1\n(2960)", 2.5, "#5dcaa5"),
    ("Router1\n(1941)", 4.5, "#afa9ec"),
    ("Router2\n(1941)", 8.5, "#afa9ec"),
    ("Switch2\n(2960)", 10.5, "#5dcaa5"),
    ("PC2\n(visualiza)", 12.5, "#85b7eb"),
]

for texto, x, color in cajas:
    ax.add_patch(patches.FancyBboxPatch((x, 1), 1.6, 1.2, boxstyle="round,pad=0.05",
                                         facecolor=color, edgecolor="black", linewidth=0.8))
    ax.text(x + 0.8, 1.6, texto, ha="center", va="center", fontsize=9)

# Conexiones cortas (UTP directo)
for x1, x2 in [(2.1, 2.5), (4.1, 4.5), (10.1, 10.5), (12.1, 12.5)]:
    ax.annotate("", xy=(x2, 1.6), xytext=(x1, 1.6),
                arrowprops=dict(arrowstyle="-", linewidth=1))

# Enlace WAN (la "nube")
ax.annotate("", xy=(8.5, 1.6), xytext=(6.1, 1.6),
            arrowprops=dict(arrowstyle="-", linewidth=1.4, linestyle="--"))
ax.text(7.3, 2.1, "enlace WAN\n(antes 'nube')", ha="center", fontsize=8, style="italic")

ax.set_xlim(0, 14.5)
ax.set_ylim(0, 3)
ax.axis("off")
plt.title("Topología física de la práctica", fontsize=11)
plt.tight_layout()
plt.show()


## 6. Direccionamiento IP

Cada tramo de la topología es su propia subred:

| Segmento | Dispositivo | IP | Máscara |
|---|---|---|---|
| LAN de PC1 | Router1 (Gig0/0) | 192.168.10.1 | /24 |
| LAN de PC1 | PC1 | 192.168.10.10 | /24 (gateway 192.168.10.1) |
| Enlace WAN | Router1 | 192.168.1.1 | /30 |
| Enlace WAN | Router2 | 192.168.1.2 | /30 |
| LAN de PC2 | Router2 (Gig0/0) | 192.168.20.1 | /24 |
| LAN de PC2 | PC2 | 192.168.20.10 | /24 (gateway 192.168.20.1) |

Rutas para que una LAN vea a la otra a través del enlace WAN:

```
Router1(config)# ip route 192.168.20.0 255.255.255.0 192.168.1.2
Router2(config)# ip route 192.168.10.0 255.255.255.0 192.168.1.1
```

Sin la ruta y sin el `gateway` correcto en cada PC, aunque la IP esté bien puesta, el tráfico nunca sale de su propia LAN.

## 7. Conexiones físicas y tipos de cable

- **PC — Switch**: cable UTP directo (patch cable normal, Cat5e o Cat6).
- **Switch — Router**: también UTP directo, hacia la interfaz LAN del router (por ejemplo `Gig0/0` en el 1941). Los routers actuales tienen auto-MDIX, así que detectan y ajustan la conexión aunque el cable no sea el "ideal".
- **Router — Router (el tramo de la nube)**:
  - Con módulos seriales (WIC-2T): cable serial V.35, un extremo DCE y otro DTE. Al lado DCE se le configura `clock rate`, porque es el que sincroniza el enlace.
  - Sin módulos seriales: un cable Ethernet directo entre las interfaces Gigabit de ambos routers, apoyándose otra vez en auto-MDIX.

En la vida real, ahí habría un proveedor de internet (ISP) con su propio equipo — la "nube" del diagrama representa justo esa parte que normalmente no controlas ni ves.

## 8. Qué hace cada PC

- **PC1**: abre `pc1_cifrado.html` de forma local (`file://`, doble clic), nunca por red. Ahí escribe el mensaje, la llave, y usa las opciones de cifrar/descifrar para preparar el texto que va a compartir.
- **PC2**: no tiene la página completa, solo `pc2_mensaje.html`. Necesita dos cosas para ver el mensaje: el texto cifrado (que le llega por el canal que hayan acordado) y la llave (acordada de antemano, no dentro del mensaje). Con esos dos datos, el descifrado ocurre en el propio navegador de PC2.

## 9. Cómo se sirven las páginas y cómo se buscan por IP

`pc1_cifrado.html` nunca se sirve por red — se queda local en PC1, por diseño, para que no exista ninguna URL por la que alguien pueda llegar a ella por error.

`pc2_mensaje.html` sí necesita estar accesible por red, porque PC2 la visita desde otra LAN. Para eso, en la PC que la va a servir (puede ser la misma PC1, en una carpeta aparte que solo contenga ese archivo):

```
cd ruta/a/servidor_pc2
python -m http.server 8080
```

Esto levanta un servidor HTTP simple en el puerto 8080, escuchando en todas las interfaces de esa máquina. Desde PC2, en el navegador, se busca directamente por IP y ruta:

```
http://192.168.20.10:8080/pc2_mensaje.html
```

No hay nombre de dominio ni DNS en este laboratorio: PC2 escribe la IP tal cual, no un nombre bonito. Si se usa el puerto 80 (requiere permisos de administrador/`sudo`), no hace falta escribir el puerto en la URL; con cualquier otro puerto sí es obligatorio.

## 10. Control de acceso (ACL)

Una ACL de Cisco filtra por IP y puerto (capa 3/4), no por ruta dentro de un sitio web — por eso la separación real entre "la página que cifra" y "la página que visualiza" se logra no sirviendo la primera por red, y no con una ACL.

Lo que la ACL sí puede hacer es limitar qué tráfico deja pasar la red de PC2 hacia el servidor: solo HTTP hacia esa IP y puerto, nada más.

```
Router2(config)# access-list 101 permit tcp 192.168.10.0 0.0.0.255 host 192.168.20.10 eq 80
Router2(config)# access-list 101 deny ip 192.168.10.0 0.0.0.255 any
Router2(config)# access-list 101 permit ip any any
Router2(config)# interface Serial0/0/0
Router2(config-if)# ip access-group 101 in
```

Nota: para recibir la página, PC2 igual tiene que enviar la petición HTTP (`GET`) — no existe forma de recibir sin pedir. Lo que la ACL logra es que esa petición sea lo único que PC2 puede mandar hacia esa red.

## 11. Resumen de la práctica

- El cifrado César desplaza cada letra `k` posiciones; descifrar es cifrar con `-k`.
- La llave se acuerda **antes**, por fuera del mensaje — por eso ambas páginas la piden como dato aparte.
- PC1 cifra localmente, sin exponer esa página en la red.
- PC2 solo visualiza: pide el texto cifrado y la llave, y descifra en su propio navegador.
- El direccionamiento IP divide la red en tres tramos (LAN de PC1, enlace WAN, LAN de PC2), unidos por rutas configuradas en los routers.
- Las conexiones físicas son UTP directo en las LAN, y serial V.35 o Ethernet directo en el tramo que representa la nube.
- `python -m http.server` sirve **la carpeta completa** donde se ejecuta — por eso solo se sirve la carpeta que contiene `pc2_mensaje.html`, nunca la que contiene `pc1_cifrado.html`.
- La ACL refuerza el filtrado por IP/puerto, pero no sustituye la decisión de no exponer la página de cifrado.